# GPU microbenchmarks

How fast this GPU computes and moves data. Each GPU test runs a few times first
(warm-up: CUDA and cuBLAS start-up is not measured), then is timed with CUDA
events and reports the **median of 10 runs**. Bandwidth counts every byte read
and written.

Needs a GPU of compute capability 7.0 or newer (T4, A100, H100, …): the image's
PyTorch has no kernels for older GPUs such as the P40.

In [ ]:
import statistics, time
import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU visible to this notebook."
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU 0: {torch.cuda.get_device_name(0)} (compute {major}.{minor})")
if major < 7:
    print("This PyTorch has no kernels for this GPU: the GPU tests below will fail. "
          "Run them on a T4 or newer.")

def gpu_time(fn, warmup=3, repeat=10):
    """Median seconds of fn() on the GPU, timed with CUDA events after a warm-up."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(repeat):
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        end.synchronize()
        times.append(start.elapsed_time(end) / 1000)
    return statistics.median(times)

def report(label, seconds, nbytes=None, flops=None):
    parts = [f"{label}: {seconds * 1000:.3f} ms"]
    if nbytes:
        parts.append(f"{nbytes / seconds / 1e9:.1f} GB/s")
    if flops:
        parts.append(f"{flops / seconds / 1e12:.2f} TFLOPS")
    print(" | ".join(parts))

def free_gpu():
    torch.cuda.empty_cache()

## Matrix-matrix multiplication
Compute-bound: FP32 on the CUDA cores, FP16 on the Tensor Cores.

In [ ]:
N = 4096
for dtype in (torch.float32, torch.float16):
    A = torch.rand(N, N, device="cuda", dtype=dtype)
    B = torch.rand(N, N, device="cuda", dtype=dtype)
    report(f"matmul {N}x{N} {str(dtype).split('.')[-1]}", gpu_time(lambda: A @ B), flops=2 * N**3)
del A, B; free_gpu()

## Vector addition
`z = x + y`: reads x and y, writes z (3 transfers per element).

In [ ]:
N = 50_000_000
x, y = torch.rand(N, device="cuda"), torch.rand(N, device="cuda")
report("vector add", gpu_time(lambda: x + y), nbytes=3 * N * x.element_size())
del x, y; free_gpu()

## Scalar-vector multiplication
`y = 3.14 * x`: reads x, writes y.

In [ ]:
N = 100_000_000
x = torch.rand(N, device="cuda")
report("scalar multiply", gpu_time(lambda: x * 3.14), nbytes=2 * N * x.element_size())
del x; free_gpu()

## Elementwise matrix addition
`C = A + B`: reads A and B, writes C.

In [ ]:
N = 8192
A, B = torch.rand(N, N, device="cuda"), torch.rand(N, N, device="cuda")
report("matrix add", gpu_time(lambda: A + B), nbytes=3 * A.nelement() * A.element_size())
del A, B; free_gpu()

## GPU memory copy (device to device)
Reads the source, writes the copy.

In [ ]:
N = 128 * 1024 * 1024
a = torch.rand(N, device="cuda")
b = torch.empty_like(a)
report("device copy", gpu_time(lambda: b.copy_(a)), nbytes=2 * a.nelement() * a.element_size())
del a, b; free_gpu()

## Host to GPU (PCIe upload)
From pinned host memory, as data loaders and vLLM copy it.

In [ ]:
N = 200_000_000
host = torch.rand(N).pin_memory()
dev = torch.empty(N, device="cuda")
report("host -> GPU", gpu_time(lambda: dev.copy_(host, non_blocking=True)), nbytes=host.nelement() * host.element_size())

## GPU to host (PCIe download)

In [ ]:
report("GPU -> host", gpu_time(lambda: host.copy_(dev, non_blocking=True)), nbytes=dev.nelement() * dev.element_size())
del host, dev; free_gpu()

## Stream triad (scale-add)
`A = A + 2.5 * B`: reads A and B, writes A.

In [ ]:
N = 50_000_000
A, B = torch.rand(N, device="cuda"), torch.rand(N, device="cuda")
report("triad", gpu_time(lambda: A.add_(B, alpha=2.5)), nbytes=3 * N * A.element_size())
del A, B; free_gpu()

## CPU matrix multiply (compute-bound, CPU)
For comparison with the GPU.

In [ ]:
N = 4000
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)
A @ B  # warm-up
times = []
for _ in range(3):
    t0 = time.perf_counter(); A @ B; times.append(time.perf_counter() - t0)
report(f"CPU matmul {N}x{N} float32", statistics.median(times), flops=2 * N**3)

## CPU memory copy (memory-bound, CPU)
Reads the source, writes the copy.

In [ ]:
N = 200_000_000
x = np.random.rand(N).astype(np.float32)
y = np.empty_like(x)
np.copyto(y, x)  # warm-up
times = []
for _ in range(3):
    t0 = time.perf_counter(); np.copyto(y, x); times.append(time.perf_counter() - t0)
report("CPU copy", statistics.median(times), nbytes=2 * x.nbytes)